In [0]:
import pyspark.sql.functions as F

catalog_name = "f1_dev"
target_table = f"{catalog_name}.silver.fact_results"



In [0]:
# 1. Read raw results table
df_results_raw = spark.table(f"{catalog_name}.bronze.results")



In [0]:
# 2. Transform & Cleanse
df_results_silver = (
    df_results_raw
    .replace(r"\N", None)
    .select(
        F.col("resultId").cast("integer").alias("result_id"),
        F.col("raceId").cast("integer").alias("race_id"),
        F.col("driverId").cast("integer").alias("driver_id"),
        F.col("constructorId").cast("integer").alias("constructor_id"),
        F.col("number").cast("integer").alias("car_number"),
        F.col("grid").cast("integer").alias("grid_position"),
        F.col("position").cast("integer").alias("position"),
        F.col("positionText").alias("position_text"),
        F.col("positionOrder").cast("integer").alias("position_order"),
        F.col("points").cast("double").alias("points"),
        F.col("laps").cast("integer").alias("laps_completed"),
        F.col("time").alias("race_time"),
        F.col("milliseconds").cast("long").alias("race_time_ms"),
        F.col("fastestLap").cast("integer").alias("fastest_lap_number"),
        F.col("rank").cast("integer").alias("fastest_lap_rank"),
        F.col("fastestLapTime").alias("fastest_lap_time"),
        F.col("fastestLapSpeed").cast("double").alias("fastest_lap_speed_kph"),
        F.col("statusId").cast("integer").alias("status_id")
    )
)



In [0]:
# 3. Write as Delta table in Silver schema
df_results_silver.write.mode("overwrite").saveAsTable(target_table)



In [0]:
# 4. Set Unity Catalog Governance Tags
spark.sql(f"""
    ALTER TABLE {target_table} SET TAGS (
        'layer' = 'silver',
        'table_type' = 'fact',
        'contains_pii' = 'false'
    )
""")

print(f"✓ Successfully built and governed: {target_table}")